In [12]:
import glob
import pickle

from music21 import converter
from music21 import instrument
from music21 import note
from music21 import chord

In [13]:
notes = []

midi_files = glob.glob(
    "maestro-v3.0.0/**/*.midi",
    recursive=True
)

print("Total MIDI Files:", len(midi_files))

Total MIDI Files: 0


In [14]:
import os

print("Current Folder:")
print(os.getcwd())

print("\nFiles/Folders:")
for item in os.listdir():
    print(item)

Current Folder:
C:\Users\acer\OneDrive\Desktop\.ipynb_checkpoints\ai music genrator

Files/Folders:
.ipynb_checkpoints
best_model.keras
maestro-v3.0.0-midi
network_input.npy
network_output.npy
notes.pkl
preprocess.ipynb
train_model.py
train_model1.ipynb


In [15]:
import os

for root, dirs, files in os.walk("maestro-v3.0.0-midi"):
    print(root)
    print("Files:", len(files))
    print("-"*50)

maestro-v3.0.0-midi
Files: 0
--------------------------------------------------
maestro-v3.0.0-midi\maestro-v3.0.0
Files: 4
--------------------------------------------------
maestro-v3.0.0-midi\maestro-v3.0.0\.ipynb_checkpoints
Files: 1
--------------------------------------------------
maestro-v3.0.0-midi\maestro-v3.0.0\2004
Files: 132
--------------------------------------------------
maestro-v3.0.0-midi\maestro-v3.0.0\2006
Files: 115
--------------------------------------------------
maestro-v3.0.0-midi\maestro-v3.0.0\2008
Files: 147
--------------------------------------------------
maestro-v3.0.0-midi\maestro-v3.0.0\2011
Files: 163
--------------------------------------------------
maestro-v3.0.0-midi\maestro-v3.0.0\2013
Files: 87
--------------------------------------------------
maestro-v3.0.0-midi\maestro-v3.0.0\2015
Files: 129
--------------------------------------------------
maestro-v3.0.0-midi\maestro-v3.0.0\2017
Files: 140
-------------------------------------------------

In [16]:
import os

midi_files = []

for root, dirs, files in os.walk("maestro-v3.0.0-midi"):
    for file in files:
        if file.endswith(".midi") or file.endswith(".mid"):
            midi_files.append(os.path.join(root,file))

print("Total MIDI Files:", len(midi_files))

if len(midi_files) > 0:
    print("\nFirst MIDI File:")
    print(midi_files[0])

Total MIDI Files: 1006

First MIDI File:
maestro-v3.0.0-midi\maestro-v3.0.0\2004\MIDI-Unprocessed_SMF_02_R1_2004_01-05_ORIG_MID--AUDIO_02_R1_2004_05_Track05_wav.midi


In [17]:
import glob

midi_files = glob.glob(
    "maestro-v3.0.0-midi/**/*.midi",
    recursive=True
)

print("Total MIDI Files:", len(midi_files))

Total MIDI Files: 1006


In [18]:
import glob

midi_files = glob.glob(
    "maestro-v3.0.0-midi/maestro-v3.0.0/**/*.midi",
    recursive=True
)

print("Total MIDI Files:", len(midi_files))

print("\nFirst 5 Files:")
for file in midi_files[:5]:
    print(file)

Total MIDI Files: 1006

First 5 Files:
maestro-v3.0.0-midi/maestro-v3.0.0\2004\MIDI-Unprocessed_SMF_02_R1_2004_01-05_ORIG_MID--AUDIO_02_R1_2004_05_Track05_wav.midi
maestro-v3.0.0-midi/maestro-v3.0.0\2004\MIDI-Unprocessed_SMF_02_R1_2004_01-05_ORIG_MID--AUDIO_02_R1_2004_06_Track06_wav.midi
maestro-v3.0.0-midi/maestro-v3.0.0\2004\MIDI-Unprocessed_SMF_02_R1_2004_01-05_ORIG_MID--AUDIO_02_R1_2004_08_Track08_wav.midi
maestro-v3.0.0-midi/maestro-v3.0.0\2004\MIDI-Unprocessed_SMF_02_R1_2004_01-05_ORIG_MID--AUDIO_02_R1_2004_10_Track10_wav.midi
maestro-v3.0.0-midi/maestro-v3.0.0\2004\MIDI-Unprocessed_SMF_05_R1_2004_01_ORIG_MID--AUDIO_05_R1_2004_02_Track02_wav.midi


In [19]:
from music21 import converter, instrument, note, chord
import os
import pickle

notes = []

for file in midi_files[:20]:

    try:

        midi = converter.parse(file)

        parts = instrument.partitionByInstrument(midi)

        if parts:
            notes_to_parse = parts.parts[0].recurse()
        else:
            notes_to_parse = midi.flat.notes

        for element in notes_to_parse:

            if isinstance(element, note.Note):
                notes.append(str(element.pitch))

            elif isinstance(element, chord.Chord):
                notes.append(
                    '.'.join(
                        str(n)
                        for n in element.normalOrder
                    )
                )

    except Exception as e:
        print("Error:", e)

print("Total Notes:", len(notes))

Total Notes: 65650


In [20]:
import pickle

with open("notes.pkl", "wb") as f:
    pickle.dump(notes, f)

print("notes.pkl saved successfully")

notes.pkl saved successfully


In [21]:
pitchnames = sorted(set(notes))

n_vocab = len(pitchnames)

print("Unique Notes:", n_vocab)

Unique Notes: 1346


In [22]:
note_to_int = dict(

    (
        note,
        number
    )

    for number,
        note

    in enumerate(
        pitchnames
    )
)

print(
    "Mapping Created"
)

Mapping Created


In [23]:
import numpy as np

In [26]:
sequence_length = 100

network_input =[]
network_output = []

for i in range(

        len(notes)

        - sequence_length):

    seq_in = notes[
        i:i+sequence_length
    ]

    seq_out = notes[
        i+sequence_length
    ]

    network_input.append(

        [
            note_to_int[n]

            for n in seq_in
        ]
    )

    network_output.append(

        note_to_int[
            seq_out
        ]
    )

print(
    "Sequences:",
    len(network_input)
) 

Sequences: 65550


In [27]:
n_patterns = len(
    network_input
)

network_input = np.reshape(

    network_input,

    (
        n_patterns,
        sequence_length,
        1
    )
)

network_input = (

    network_input

    / float(
        n_vocab
    )
)

print(
    network_input.shape
)

(65550, 100, 1)


In [28]:
from tensorflow.keras.utils import to_categorical

network_output = to_categorical(
    network_output
)

print(network_output.shape)

(65550, 1346)


In [29]:
import numpy as np

np.save(
    "network_input.npy",
    network_input
)

np.save(
    "network_output.npy",
    network_output
)

print("Training Data Saved")

Training Data Saved


In [30]:
print("Total Notes:", len(notes))
print("Unique Notes:", n_vocab)
print("Input Shape:", network_input.shape)
print("Output Shape:", network_output.shape)

Total Notes: 65650
Unique Notes: 1346
Input Shape: (65550, 100, 1)
Output Shape: (65550, 1346)
